In [ ]:
# Install qqman if you haven't already
# install.packages("qqman")
# install.packages("data.table")
# Load the package
library(tidyverse)
library(ggplot2)
library(data.table)
library(purrr)

In [ ]:
# Remove directories utlized in this script to ensure clean workspace 
# Recrate directories utlized in this script
system("rm -rf saige_gene_plot", intern=TRUE)
system("mkdir -p saige_gene_plot", intern=TRUE)

In [ ]:

if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("biomaRt")

install.packages("openxlsx")

install.packages("paletteer")



In [ ]:
# Install qqman if you haven't already
# install.packages("qqman")
# install.packages("data.table")
# Load the package
library(tidyverse)
library(ggplot2)
library(data.table)
library(janitor)
library(flextable)
library(officer)
library(openxlsx)
library(biomaRt)


In [ ]:
data <- fread("merged_saige_gene_step02/EUR_aml_phenotype_high_confidence_merged_chr")
# Connect to Ensembl
mart <- useMart("ensembl", dataset = "hsapiens_gene_ensembl")
# Loop through unique Ensembl IDs
gene_info <- getBM(
    attributes = c("ensembl_gene_id", "hgnc_symbol"),
    filters = "ensembl_gene_id",
    values = unique(data$Region),
    mart = mart
  )

In [ ]:
gene_info <- 
rename(gene_info, 
       "Region" = "ensembl_gene_id", 
       "Symbol" = "hgnc_symbol" )

In [ ]:

# Directory containing gene set-based rare variant assoication test results 
files <- c("EUR_aml_phenotype_high_confidence_merged_chr", "EUR_aml_phenotype_synonymous_merged_chr")
# Load file containing conversion from Ensembl gene IDs to HUGO gene symbols file 
# gene_symbols <- rename(fread("gene_symbols.tsv"), "Region" = "gene_id")

# Function: Formats the gene set-based rare variant assoication test results 
process_results <- function(results) { 
    results %>% 
    filter(Group != "Cauchy") %>% 
    # inner_join(gene_symbols, by = "Region") %>% 
    mutate(max_MAF = as.character(max_MAF)) %>% 
    mutate(negative_log_p_values = -log10(Pvalue)) %>% 
    mutate(Chromosome = gsub("chr", "", Chromosome)) %>% 
    mutate(Chromosome = as.numeric(Chromosome)) %>% 
    mutate(Region = as.factor(Region)) %>% 
    mutate(Region_factor = fct_reorder(Region, Chromosome)) %>% 
    full_join(gene_info, by = "Region")
  }

# Loop: Loads and format files 
for (f in files) {
  print(f)
  data <- fread(paste0("merged_saige_gene_step02/", f))
  processed <- process_results(data)
  assign(f, processed, envir = .GlobalEnv)
}

In [ ]:
# Create a list of gene set-based rare variant assoication test results 
df_list <- list(EUR_aml_phenotype_high_confidence = EUR_aml_phenotype_high_confidence_merged_chr)

In [ ]:
# Set encode colors for the plots 
color_values <- c("01" = "#FCCC87FF",
                  "02" = "#B37A32FF",
                  "03" = "#8f5a16ff",
                  "11" = "#3FA3C5FF",
                  "12" = "#0F7BA2FF",
                  "13" = "#0B5A79FF")

In [ ]:
paletteer::paletteer_d("LaCroixColoR::Lemon")

In [ ]:
# Set encode colors for the plots 
color_values <- paletteer::paletteer_d("LaCroixColoR::Lemon")

In [ ]:
# Loop: Manhattan plot for meta-analyzed gene set-based rare variant association test results without polygenic risk score
for (df_name in names(df_list)) {
  
  df <- df_list[[df_name]]
  
  # Set Bonferroni significance level
  significance_value <- -log10(0.05 / nrow(df))
  
  # Ensure Chromosome is numeric for sorting
  df$Chromosome <- as.numeric(as.character(df$Chromosome))
  
  # Compute cumulative x-axis positions for Manhattan-style plotting
  df <- df %>%
    arrange(Chromosome) %>%
    group_by(Chromosome) %>%
    mutate(region_index = row_number()) %>%
    ungroup()
  
  chrom_sizes <- df %>%
    group_by(Chromosome) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(cum_start = lag(cumsum(n), default = 0))
  
  df <- df %>%
    left_join(chrom_sizes %>% dplyr::select(Chromosome, cum_start), by = "Chromosome") %>%
    mutate(x_cum = cum_start + region_index)
  
  # Chromosome midpoints for x-axis labels
  chrom_mid <- df %>%
    group_by(Chromosome) %>%
    summarise(mid = mean(x_cum), .groups = "drop")
  
  # Encode colors and MAF order
  df <- df %>%
    mutate(
      color = as.factor(Chromosome %% 2),
      maf_order = case_when(
        max_MAF == "1e-04" ~ 1,
        max_MAF == "0.001" ~ 2,
        max_MAF == "0.01" ~ 3,
        TRUE ~ NA_real_
      ),
      color_code = as.factor(paste0(color, maf_order))
    ) %>%
    filter(max_MAF == "1e-04")  # Filter MAF after x_cum is computed
  
  # Get top 10 hits by -log10(p-value)
  top_values <- df %>% arrange(-negative_log_p_values) %>% slice(1:10)
  
  # Plot
  plot <- ggplot(df, aes(x = x_cum, y = negative_log_p_values)) +
    geom_point(aes(color = color_code, size = negative_log_p_values, shape = Group)) +
    scale_shape_discrete(labels = c("LoF", "Missense", "LoF+Missense")) +
    # Define your colors here
    scale_color_manual(values = color_values, labels = c("1 x 10⁴", "1 x 10⁴")) +
    labs(y = "-log10(p-values)", shape = "Functional Group", color = "Minor Allele Frequency") +
    geom_hline(yintercept = significance_value, color = "#DD5129FF", linetype = "dashed", linewidth = 1.5) +
    theme_bw() +
    theme(
      axis.title.x = element_blank(),
      legend.key = element_rect(fill = "white"),
      legend.text = element_text(size = 12, face = "italic"),
      legend.title = element_text(size = 10),
      plot.title = element_text(size = 22),
      axis.title.y = element_text(size = 16),
      axis.text.y = element_text(size = 14),
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank()
    ) +
    scale_x_continuous(breaks = chrom_mid$mid, labels = chrom_mid$Chromosome) +
    guides(size = "none",
           color = guide_legend(override.aes = list(size = 5)),
           shape = guide_legend(override.aes = list(size = 5))) +
    geom_text(
      data = top_values,
      aes(x = x_cum, y = negative_log_p_values, label = Symbol),
      check_overlap = TRUE, vjust = 0.5, hjust = -0.2
    )
  
  # Save plot
  ggsave(
    filename = paste0("saige_gene_plot/manhattan_", df_name, "_maf_0.0001_plot.png"), 
    plot = plot, width = 20, height = 10, dpi = 300
  )
  
}


In [ ]:
# Loop: Manhattan plot for meta-analyzed gene set-based rare variant association test results without polygenic risk score
for (df_name in names(df_list)) {
  
  df <- df_list[[df_name]]
  
  # Set Bonferroni significance level
  significance_value <- -log10(0.05 / nrow(df))
  
  # Ensure Chromosome is numeric for sorting
  df$Chromosome <- as.numeric(as.character(df$Chromosome))
  
  # Compute cumulative x-axis positions for Manhattan-style plotting
  df <- df %>%
    arrange(Chromosome) %>%
    group_by(Chromosome) %>%
    mutate(region_index = row_number()) %>%
    ungroup()
  
  chrom_sizes <- df %>%
    group_by(Chromosome) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(cum_start = lag(cumsum(n), default = 0))
  
  df <- df %>%
    left_join(chrom_sizes %>% dplyr::select(Chromosome, cum_start), by = "Chromosome") %>%
    mutate(x_cum = cum_start + region_index)
  
  # Chromosome midpoints for x-axis labels
  chrom_mid <- df %>%
    group_by(Chromosome) %>%
    summarise(mid = mean(x_cum), .groups = "drop")
  
  # Encode colors and MAF order
  df <- df %>%
    mutate(
      color = as.factor(Chromosome %% 2),
      maf_order = case_when(
        max_MAF == "1e-04" ~ 1,
        max_MAF == "0.001" ~ 2,
        max_MAF == "0.01" ~ 3,
        TRUE ~ NA_real_
      ),
      color_code = as.factor(paste0(color, maf_order))
    ) %>%
    filter(max_MAF == "0.001")  # Filter MAF after x_cum is computed
  
  # Get top 10 hits by -log10(p-value)
  top_values <- df %>% arrange(-negative_log_p_values) %>% slice(1:10)
  
  # Plot
  plot <- ggplot(df, aes(x = x_cum, y = negative_log_p_values)) +
    geom_point(aes(color = color_code, size = negative_log_p_values, shape = Group)) +
    scale_shape_discrete(labels = c("LoF", "Missense", "LoF+Missense")) +
    # Define your colors here
    scale_color_manual(values = color_values, labels = c("1 x 10³", "1 x 10³")) +
    labs(y = "-log10(p-values)", shape = "Functional Group", color = "Minor Allele Frequency") +
    geom_hline(yintercept = significance_value, color = "#DD5129FF", linetype = "dashed", linewidth = 1.5) +
    theme_bw() +
    theme(
      axis.title.x = element_blank(),
      legend.key = element_rect(fill = "white"),
      legend.text = element_text(size = 12, face = "italic"),
      legend.title = element_text(size = 10),
      plot.title = element_text(size = 22),
      axis.title.y = element_text(size = 16),
      axis.text.y = element_text(size = 14),
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank()
    ) +
    scale_x_continuous(breaks = chrom_mid$mid, labels = chrom_mid$Chromosome) +
    guides(size = "none",
           color = guide_legend(override.aes = list(size = 5)),
           shape = guide_legend(override.aes = list(size = 5))) +
    geom_text(
      data = top_values,
      aes(x = x_cum, y = negative_log_p_values, label = Symbol),
      check_overlap = TRUE, vjust = 0.5, hjust = -0.2
    )
  
  # Save plot
  ggsave(
    filename = paste0("saige_gene_plot/manhattan_", df_name, "_maf_0.001_plot.png"), 
    plot = plot, width = 20, height = 10, dpi = 300
  )
  
}


In [ ]:
# Loop: Manhattan plot for meta-analyzed gene set-based rare variant association test results without polygenic risk score
for (df_name in names(df_list)) {
  
  df <- df_list[[df_name]]
  
  # Set Bonferroni significance level
  significance_value <- -log10(0.05 / nrow(df))
  
  # Ensure Chromosome is numeric for sorting
  df$Chromosome <- as.numeric(as.character(df$Chromosome))
  
  # Compute cumulative x-axis positions for Manhattan-style plotting
  df <- df %>%
    arrange(Chromosome) %>%
    group_by(Chromosome) %>%
    mutate(region_index = row_number()) %>%
    ungroup()
  
  chrom_sizes <- df %>%
    group_by(Chromosome) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(cum_start = lag(cumsum(n), default = 0))
  
  df <- df %>%
    left_join(chrom_sizes %>% dplyr::select(Chromosome, cum_start), by = "Chromosome") %>%
    mutate(x_cum = cum_start + region_index)
  
  # Chromosome midpoints for x-axis labels
  chrom_mid <- df %>%
    group_by(Chromosome) %>%
    summarise(mid = mean(x_cum), .groups = "drop")
  
  # Encode colors and MAF order
  df <- df %>%
    mutate(
      color = as.factor(Chromosome %% 2),
      maf_order = case_when(
        max_MAF == "1e-04" ~ 1,
        max_MAF == "0.001" ~ 2,
        max_MAF == "0.01" ~ 3,
        TRUE ~ NA_real_
      ),
      color_code = as.factor(paste0(color, maf_order))
    ) %>%
    filter(max_MAF == "0.01")  # Filter MAF after x_cum is computed
  
  # Get top 10 hits by -log10(p-value)
  top_values <- df %>% arrange(-negative_log_p_values) %>% slice(1:10)
  
  # Plot
  plot <- ggplot(df, aes(x = x_cum, y = negative_log_p_values)) +
    geom_point(aes(color = color_code, size = negative_log_p_values, shape = Group)) +
    scale_shape_discrete(labels = c("LoF", "Missense", "LoF+Missense")) +
    # Define your colors here
    scale_color_manual(values = color_values,  labels = c("1 x 10²","1 x 10²")) +
    labs(y = "-log10(p-values)", shape = "Functional Group", color = "Minor Allele Frequency") +
    geom_hline(yintercept = significance_value, color = "#DD5129FF", linetype = "dashed", linewidth = 1.5) +
    theme_bw() +
    theme(
      axis.title.x = element_blank(),
      legend.key = element_rect(fill = "white"),
      legend.text = element_text(size = 12, face = "italic"),
      legend.title = element_text(size = 10),
      plot.title = element_text(size = 22),
      axis.title.y = element_text(size = 16),
      axis.text.y = element_text(size = 14),
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank()
    ) +
    scale_x_continuous(breaks = chrom_mid$mid, labels = chrom_mid$Chromosome) +
    guides(size = "none",
           color = guide_legend(override.aes = list(size = 5)),
           shape = guide_legend(override.aes = list(size = 5))) +
    geom_text(
      data = top_values,
      aes(x = x_cum, y = negative_log_p_values, label = Symbol),
      check_overlap = TRUE, vjust = 0.5, hjust = -0.2
    )
  
  # Save plot
  ggsave(
    filename = paste0("saige_gene_plot/manhattan_", df_name, "_maf_0.01_plot.png"), 
    plot = plot, width = 20, height = 10, dpi = 300
  )
  
}
